In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import shutil
import random
import pandas as pd
from pathlib import Path
import zipfile

def get_participant_id_from_name(name):
  
    base_name = os.path.splitext(name)[0]
    parts = base_name.split('_')
    if len(parts) >= 2 and parts[0].upper() == 'P' and parts[1].isdigit():
        return 'P' + parts[1] 
    return None

def split_dataset_with_labels(source_dir, label_files, output_dir, n_participants=22, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
   
    random.seed(seed)

    # Verify ratios sum to 1
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, 

    # Read labels from Excel files
    print("Reading labels from Excel files...")
    labels_dfs = {}
    folder_mapping = {}  

    for folder_name, label_file in label_files.items():
        if os.path.exists(label_file):
            labels_dfs[folder_name] = pd.read_excel(label_file)
            print(f"  {folder_name}: Loaded {len(labels_dfs[folder_name])} rows")
            print(f"    Columns: {labels_dfs[folder_name].columns.tolist()}")
        else:
            print(f"  WARNING: {folder_name} label file not found: {label_file}")
            labels_dfs[folder_name] = pd.DataFrame()

    # Get all participant directories 
    source_path = Path(source_dir)
    root_folders = [d for d in source_path.iterdir() if d.is_dir()]

    print(f"\nFound root folders: {[d.name for d in root_folders]}")

    # Create mapping between label file names and actual folder names (case-insensitive)
    for folder_dir in root_folders:
        folder_lower = folder_dir.name.lower()
        for label_key in label_files.keys():
            if label_key.lower() in folder_lower or folder_lower in label_key.lower():
                folder_mapping[label_key] = folder_dir.name
                print(f"  Mapping: '{label_key}' -> '{folder_dir.name}'")
                break

    # Define all participants P001 to P0XX based on n_participants
    all_participants = [f'P{str(i).zfill(3)}' for i in range(1, n_participants + 1)]
    print(f"\nTotal participants (P001-P{str(n_participants).zfill(3)}): {all_participants}")

    # Randomly shuffle participants 
    participants_shuffled = all_participants.copy()
    random.shuffle(participants_shuffled)

    # Calculate split points
    n_total = len(all_participants)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    

    print(f"Split distribution: Train={n_train}, Val={n_val}, Test={n_total - n_train - n_val}")

    # Split participants randomly
    train_participants = sorted(participants_shuffled[:n_train])
    val_participants = sorted(participants_shuffled[n_train:n_train + n_val])
    test_participants = sorted(participants_shuffled[n_train + n_val:])

    print(f"\nRandomly selected participants:")
    print(f"Train ({len(train_participants)}): {train_participants}")
    print(f"Val   ({len(val_participants)}): {val_participants}")
    print(f"Test  ({len(test_participants)}): {test_participants}")

    # Create output directories
    output_path = Path(output_dir)
    splits = {
        'train': train_participants,
        'val': val_participants,
        'test': test_participants
    }

    # Split labels and copy images
    split_labels = {split_name: {} for split_name in splits.keys()}

    for split_name, participant_list in splits.items():
        print(f"\n{'='*60}")
        print(f"Processing {split_name.upper()} set...")
        print(f"{'='*60}")
        print(f"  DEBUG: current participant_list for {split_name}: {participant_list}")

        split_dir = output_path / split_name
        split_dir.mkdir(parents=True, exist_ok=True)

        # Copy all images and folders for participants in this split
        total_img_count = 0

        for root_folder in root_folders:
            actual_folder_name = root_folder.name  
            folder_img_count = 0

            # Find the corresponding label key 
            label_key = None
            for key, mapped_name in folder_mapping.items():
                if mapped_name == actual_folder_name:
                    label_key = key
                    break

            for participant_dir in root_folder.iterdir():
                # Extract participant ID from the directory name for comparison
                extracted_p_id = get_participant_id_from_name(participant_dir.name)

                print(f"  DEBUG: Checking participant_dir: {participant_dir.name} (full path: {participant_dir})")
                print(f"    is_dir: {participant_dir.is_dir()}, Extracted P_ID: {extracted_p_id}, in_participant_list: {extracted_p_id in participant_list}")
                print(f"    DEBUG: participant_dir.name: {repr(participant_dir.name)} (type: {type(participant_dir.name)}) ")
                if participant_list:
                    print(f"    DEBUG: participant_list first element: {repr(participant_list[0])} (type: {type(participant_list[0])})")

                if participant_dir.is_dir() and extracted_p_id in participant_list:
                    # Recreate the structure: split/root_folder/participant/task/box/
                    dest_root_dir = split_dir / actual_folder_name
                    dest_participant_dir = dest_root_dir / participant_dir.name

                    # Copy entire participant directory structure
                    print(f"  DEBUG: Attempting to copy from {participant_dir} to {dest_participant_dir}")
                    if dest_participant_dir.exists():
                        shutil.rmtree(dest_participant_dir)
                    shutil.copytree(participant_dir, dest_participant_dir)
                    print(f"  DEBUG: Successfully copied {participant_dir.name}")

                    # Count images
                    found_files_in_walk = []
                    for root, dirs, files in os.walk(dest_participant_dir):
                        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
                        folder_img_count += len(image_files)
                        if image_files:
                            found_files_in_walk.append(f"  DEBUG: Found {len(image_files)} images in {root.replace(str(dest_participant_dir), '') or '/'}: {image_files[:3]}...")

                    if found_files_in_walk:
                        for msg in found_files_in_walk:
                            print(msg)
                    else:
                        print(f"  DEBUG: No image files found during os.walk in {dest_participant_dir}")

                    print(f"  Copied: {actual_folder_name}/{participant_dir.name}")

            print(f"  {actual_folder_name}: {folder_img_count} images")
            total_img_count += folder_img_count

            # Filter labels for this folder and split (using the label_key)
            if label_key and label_key in labels_dfs and not labels_dfs[label_key].empty:
                # Try to find participant ID column or extract from Image Path
                participant_col = None
                df = labels_dfs[label_key]
                print(f"  DEBUG: Processing labels for {label_key}. First 5 Image Paths: {df['Image Path'].head().tolist() if 'Image Path' in df.columns else 'N/A'}")

                # First, try to find a direct participant ID column
                for col in df.columns:
                    if 'participant' in col.lower() or 'patient' in col.lower() or 'id' in col.lower():
                        participant_col = col
                        break

                # If no direct column, extract from Image Path
                if participant_col is None and 'Image Path' in df.columns:
                    
                    df['Participant_ID'] = df['Image Path'].apply(
                        lambda x: get_participant_id_from_name(x.split('/')[0]) if isinstance(x, str) else None
                    )
                    participant_col = 'Participant_ID'
                    print(f"  Extracted participant IDs from Image Path for {label_key}")
                    print(f"  DEBUG: First 5 extracted Participant_IDs: {df['Participant_ID'].head().tolist()}")

                if participant_col:
                    split_labels[split_name][label_key] = df[
                        df[participant_col].isin(participant_list)
                    ].copy()
                    print(f"  {label_key} labels: {len(split_labels[split_name][label_key])} rows filtered")
                else:
                    print(f"  WARNING: Could not find or extract participant ID in {label_key} labels")
                    split_labels[split_name][label_key] = pd.DataFrame()

        print(f"\n{split_name} TOTAL: {len(participant_list)} participants, {total_img_count} images copied")

        # Save labels for each folder in this split
        for label_key in labels_dfs.keys():
            if label_key in split_labels[split_name] and not split_labels[split_name][label_key].empty:
                label_output_path = output_path / f'{split_name}_{label_key}_labels.xlsx'
                split_labels[split_name][label_key].to_excel(label_output_path, index=False)
                print(f"Saved: {label_output_path}")

    # Save split information
    split_info_path = output_path / 'split_info.txt'
    with open(split_info_path, 'w') as f:
        f.write(f"Dataset Split Information (seed={seed})\n")
        f.write("="*60 + "\n\n")
        f.write(f"TRAIN PARTICIPANTS ({len(train_participants)}):\n")
        f.write(', '.join(train_participants) + "\n\n")
        f.write(f"VALIDATION PARTICIPANTS ({len(val_participants)}):\n")
        f.write(', '.join(val_participants) + "\n\n")
        f.write(f"TEST PARTICIPANTS ({len(test_participants)}):\n")
        f.write(', '.join(test_participants) + "\n")

    print(f"\n{'='*60}")
    print(f"Dataset split complete!")
    print(f"Output directory: {output_dir}")
    print(f"Split information: {split_info_path}")
    print(f"{'='*60}")

    return {
        'train': {'participants': train_participants, 'labels': split_labels['train']},
        'val': {'participants': val_participants, 'labels': split_labels['val']},
        'test': {'participants': test_participants, 'labels': split_labels['test']}
    }


def create_zip_files(output_dir, cleanup=True):
    
    output_path = Path(output_dir)
    zip_files = []

    for split in ['train', 'val', 'test']:
        split_dir = output_path / split
        if split_dir.exists():
            zip_filename = output_path / f'{split}_dataset.zip'

            print(f"\nCreating {split}_dataset.zip...")
            with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                # Add all files in the split directory
                for root, dirs, files in os.walk(split_dir):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, output_path)
                        zipf.write(file_path, arcname)

                # Add the corresponding label files
                for label_file in output_path.glob(f'{split}_*_labels.xlsx'):
                    zipf.write(label_file, label_file.name)

            zip_files.append(str(zip_filename))
            print(f"  Created: {zip_filename}")

    # Also create split_info.txt zip or add to each
    split_info = output_path / 'split_info.txt'
    if split_info.exists():
        for zip_file in zip_files:
            with zipfile.ZipFile(zip_file, 'a') as zipf:
                zipf.write(split_info, 'split_info.txt')

    print(f"\n{'='*60}")
    print(f"Created {len(zip_files)} zip files:")
    for zf in zip_files:
        size_mb = os.path.getsize(zf) / (1024 * 1024)
        print(f"  {os.path.basename(zf)}: {size_mb:.2f} MB")
    print(f"{'='*60}")

    # Cleanup: Remove folders and Excel files, keep only zip files
    if cleanup:
        print("\nCleaning up (removing folders and Excel files, keeping only zips)...")
        for split in ['train', 'val', 'test']:
            split_dir = output_path / split
            if split_dir.exists():
                shutil.rmtree(split_dir)
                print(f"  Removed: {split}/ folder")

        # Remove Excel label files
        for excel_file in output_path.glob('*_labels.xlsx'):
            excel_file.unlink()
            print(f"  Removed: {excel_file.name}")

        print("\nOnly zip files remain in output directory!")

    return zip_files



Reading labels from Excel files...
  letter: Loaded 1232 rows
    Columns: ['Image Path', 'Label']
  word: Loaded 220 rows
    Columns: ['Image Path', 'Label']
  paragraph: Loaded 22 rows
    Columns: ['Image Path', 'Label']

Found root folders: ['Letters', 'Paragraphs', 'Words']
  Mapping: 'letter' -> 'Letters'
  Mapping: 'paragraph' -> 'Paragraphs'
  Mapping: 'word' -> 'Words'

Total participants (P001-P022): ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P010', 'P011', 'P012', 'P013', 'P014', 'P015', 'P016', 'P017', 'P018', 'P019', 'P020', 'P021', 'P022']
Split distribution: Train=17, Val=2, Test=3

Randomly selected participants:
Train (17): ['P002', 'P003', 'P005', 'P006', 'P007', 'P010', 'P011', 'P012', 'P013', 'P014', 'P015', 'P016', 'P017', 'P018', 'P019', 'P020', 'P022']
Val   (2): ['P008', 'P009']
Test  (3): ['P001', 'P004', 'P021']

Processing TRAIN set...
  DEBUG: current participant_list for train: ['P002', 'P003', 'P005', 'P006', 'P007', 'P010',

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import random
import pandas as pd
from pathlib import Path
import zipfile
import re

def get_participant_id_from_name(name):
    """Extracts P_XXX format participant ID from directory name or image path."""
    # Handle formats like: P_001, P001, P_001_task1, etc.
    # Use regex to find P followed by digits (with or without underscore)
    match = re.search(r'P[_]?(\d+)', name, re.IGNORECASE)
    if match:
        # Extract the number and format as P_001, P_002, etc.
        number = match.group(1)
        return f'P_{number.zfill(3)}'  # Returns P_001, P_002, etc.
    return None

def split_dataset_with_labels(source_dir, label_files, output_dir, n_participants=22, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    """
    Split dataset by participants into train/val/test sets with labels.

    Args:
        source_dir: Path to source directory with structure:
                    - Letters/P_001_xxx/images or Letters/P_001/images
                    - Words/P_001_xxx/images or Words/P_001/images
                    - Paragraphs/P_001_xxx/images or Paragraphs/P_001/images
        label_files: Dictionary with label file paths for each folder
        output_dir: Path to output directory
        n_participants: Total number of participants (P_001 to P_0XX), default 22
        train_ratio, val_ratio, test_ratio: Split ratios
        seed: Random seed for reproducibility
    """
    random.seed(seed)
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"

    # Read labels from Excel files
    print("Reading labels from Excel files...")
    labels_dfs = {}
    folder_mapping = {}

    for folder_name, label_file in label_files.items():
        if os.path.exists(label_file):
            labels_dfs[folder_name] = pd.read_excel(label_file)
            print(f"  {folder_name}: Loaded {len(labels_dfs[folder_name])} rows")
            print(f"    Columns: {labels_dfs[folder_name].columns.tolist()}")
        else:
            print(f"  WARNING: {folder_name} label file not found: {label_file}")
            labels_dfs[folder_name] = pd.DataFrame()

    # Get all participant directories
    source_path = Path(source_dir)
    root_folders = [d for d in source_path.iterdir() if d.is_dir()]

    print(f"\nFound root folders: {[d.name for d in root_folders]}")

    # Create mapping between label file names and actual folder names
    for folder_dir in root_folders:
        folder_lower = folder_dir.name.lower()
        for label_key in label_files.keys():
            if label_key.lower() in folder_lower or folder_lower in label_key.lower():
                folder_mapping[label_key] = folder_dir.name
                print(f"  Mapping: '{label_key}' -> '{folder_dir.name}'")
                break

    # Define all participants P_001 to P_0XX
    all_participants = [f'P_{str(i).zfill(3)}' for i in range(1, n_participants + 1)]
    print(f"\nTotal participants (P_001-P_{str(n_participants).zfill(3)}): {all_participants}")

    # Randomly shuffle participants
    participants_shuffled = all_participants.copy()
    random.shuffle(participants_shuffled)

    # Calculate split points
    n_total = len(all_participants)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    print(f"Split distribution: Train={n_train}, Val={n_val}, Test={n_total - n_train - n_val}")

    # Split participants randomly
    train_participants = sorted(participants_shuffled[:n_train])
    val_participants = sorted(participants_shuffled[n_train:n_train + n_val])
    test_participants = sorted(participants_shuffled[n_train + n_val:])

    print(f"\nRandomly selected participants:")
    print(f"Train ({len(train_participants)}): {train_participants}")
    print(f"Val   ({len(val_participants)}): {val_participants}")
    print(f"Test  ({len(test_participants)}): {test_participants}")

    # Create output directories
    output_path = Path(output_dir)
    splits = {
        'train': train_participants,
        'val': val_participants,
        'test': test_participants
    }

    # Split labels and copy images
    split_labels = {split_name: {} for split_name in splits.keys()}

    for split_name, participant_list in splits.items():
        print(f"\n{'='*60}")
        print(f"Processing {split_name.upper()} set...")
        print(f"{'='*60}")

        split_dir = output_path / split_name
        split_dir.mkdir(parents=True, exist_ok=True)

        # Copy all images and folders for participants in this split
        total_img_count = 0

        for root_folder in root_folders:
            actual_folder_name = root_folder.name
            folder_img_count = 0

            # Find the corresponding label key
            label_key = None
            for key, mapped_name in folder_mapping.items():
                if mapped_name == actual_folder_name:
                    label_key = key
                    break

            for participant_dir in root_folder.iterdir():
                # Handle two cases:
                # 1. Directory format: Letters/P_001_task1/ or Words/P_001_task2/
                # 2. File format: Paragraphs/P_001_task3.png (images directly in folder)

                if participant_dir.is_dir():
                    # Case 1: Participant has a directory
                    extracted_p_id = get_participant_id_from_name(participant_dir.name)

                    if extracted_p_id and extracted_p_id in participant_list:
                        # Recreate structure: split/root_folder/participant/
                        dest_root_dir = split_dir / actual_folder_name
                        dest_participant_dir = dest_root_dir / participant_dir.name

                        # Copy entire participant directory
                        if dest_participant_dir.exists():
                            shutil.rmtree(dest_participant_dir)
                        shutil.copytree(participant_dir, dest_participant_dir)

                        # Count images
                        for root, dirs, files in os.walk(dest_participant_dir):
                            folder_img_count += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])

                        print(f"  Copied: {actual_folder_name}/{participant_dir.name} (ID: {extracted_p_id})")

                elif participant_dir.is_file():
                    # Case 2: Image files directly in folder (like Paragraphs)
                    if participant_dir.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']:
                        extracted_p_id = get_participant_id_from_name(participant_dir.name)

                        if extracted_p_id and extracted_p_id in participant_list:
                            # Create destination directory for this participant
                            dest_root_dir = split_dir / actual_folder_name
                            dest_root_dir.mkdir(parents=True, exist_ok=True)

                            # Copy the image file
                            dest_file = dest_root_dir / participant_dir.name
                            shutil.copy2(participant_dir, dest_file)
                            folder_img_count += 1

            # Show summary for this folder
            if folder_img_count > 0:
                print(f"  {actual_folder_name}: {folder_img_count} images copied")
            else:
                print(f"  {actual_folder_name}: 0 images")
            total_img_count += folder_img_count

            # Filter labels for this folder and split
            if label_key and label_key in labels_dfs and not labels_dfs[label_key].empty:
                df = labels_dfs[label_key].copy()

                # Extract participant ID from Image Path
                if 'Image Path' in df.columns:
                    df['Participant_ID'] = df['Image Path'].apply(
                        lambda x: get_participant_id_from_name(str(x)) if pd.notna(x) else None
                    )

                    # Filter for current split
                    split_labels[split_name][label_key] = df[
                        df['Participant_ID'].isin(participant_list)
                    ].copy()

                    print(f"  {label_key} labels: {len(split_labels[split_name][label_key])} rows filtered")
                else:
                    print(f"  WARNING: No 'Image Path' column in {label_key} labels")
                    split_labels[split_name][label_key] = pd.DataFrame()

        print(f"\n{split_name} TOTAL: {len(participant_list)} participants, {total_img_count} images copied")

        # Save labels for each folder in this split
        for label_key in labels_dfs.keys():
            if label_key in split_labels[split_name] and not split_labels[split_name][label_key].empty:
                label_output_path = output_path / f'{split_name}_{label_key}_labels.xlsx'
                split_labels[split_name][label_key].to_excel(label_output_path, index=False)
                print(f"Saved: {label_output_path}")

    # Save split information
    split_info_path = output_path / 'split_info.txt'
    with open(split_info_path, 'w') as f:
        f.write(f"Dataset Split Information (seed={seed})\n")
        f.write("="*60 + "\n\n")
        f.write(f"TRAIN PARTICIPANTS ({len(train_participants)}):\n")
        f.write(', '.join(train_participants) + "\n\n")
        f.write(f"VALIDATION PARTICIPANTS ({len(val_participants)}):\n")
        f.write(', '.join(val_participants) + "\n\n")
        f.write(f"TEST PARTICIPANTS ({len(test_participants)}):\n")
        f.write(', '.join(test_participants) + "\n")

    print(f"\n{'='*60}")
    print(f"Dataset split complete!")
    print(f"Output directory: {output_dir}")
    print(f"Split information: {split_info_path}")
    print(f"{'='*60}")

    return {
        'train': {'participants': train_participants, 'labels': split_labels['train']},
        'val': {'participants': val_participants, 'labels': split_labels['val']},
        'test': {'participants': test_participants, 'labels': split_labels['test']}
    }


def create_zip_files(output_dir, cleanup=False):
    """Create separate zip files for train, val, and test datasets."""
    output_path = Path(output_dir)
    zip_files = []

    for split in ['train', 'val', 'test']:
        split_dir = output_path / split
        if split_dir.exists():
            zip_filename = output_path / f'{split}_dataset.zip'

            print(f"\nCreating {split}_dataset.zip...")
            with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                # Add all files in the split directory
                for root, dirs, files in os.walk(split_dir):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, output_path)
                        zipf.write(file_path, arcname)

                # Add the corresponding label files
                for label_file in output_path.glob(f'{split}_*_labels.xlsx'):
                    zipf.write(label_file, label_file.name)

            zip_files.append(str(zip_filename))
            print(f"  Created: {zip_filename}")

    # Add split_info.txt to each zip
    split_info = output_path / 'split_info.txt'
    if split_info.exists():
        for zip_file in zip_files:
            with zipfile.ZipFile(zip_file, 'a') as zipf:
                zipf.write(split_info, 'split_info.txt')

    print(f"\n{'='*60}")
    print(f"Created {len(zip_files)} zip files:")
    for zf in zip_files:
        size_mb = os.path.getsize(zf) / (1024 * 1024)
        print(f"  {os.path.basename(zf)}: {size_mb:.2f} MB")
    print(f"{'='*60}")

    if cleanup:
        print("\nCleaning up (removing folders and Excel files)...")
        for split in ['train', 'val', 'test']:
            split_dir = output_path / split
            if split_dir.exists():
                shutil.rmtree(split_dir)
                print(f"  Removed: {split}/ folder")

        for excel_file in output_path.glob('*_labels.xlsx'):
            excel_file.unlink()
            print(f"  Removed: {excel_file.name}")

        print("\nOnly zip files remain!")

    return zip_files


# Example usage:
if __name__ == "__main__":
    source_directory = "/content/drive/MyDrive/Sample_Croped_dataset"

    label_excel_files = {
        'letter': '/content/drive/MyDrive/Sample_Croped_dataset/Letters/task1_labels.xlsx',
        'word': '/content/drive/MyDrive/Sample_Croped_dataset/Words/task2_labels.xlsx',
        'paragraph': '/content/drive/MyDrive/Sample_Croped_dataset/Paragraphs/task3_labels.xlsx'
    }

    output_directory = "/content/split_datasets"  # Fixed: added leading slash

    # Split dataset
    results = split_dataset_with_labels(
        source_dir=source_directory,
        label_files=label_excel_files,
        output_dir=output_directory,
        n_participants=22,
        train_ratio=0.8,
        val_ratio=0.1,
        test_ratio=0.1,
        seed=42
    )

    print(f"\nFinal Summary:")
    print(f"Train participants: {results['train']['participants']}")
    print(f"Val participants: {results['val']['participants']}")
    print(f"Test participants: {results['test']['participants']}")

    # Create zip files
    print("\n" + "="*60)
    print("Creating zip files for download...")
    print("="*60)
    zip_files = create_zip_files(output_directory, cleanup=False)

    # Download in Colab
    try:
        from google.colab import files
        print("\nDownloading zip files...")
        for zip_file in zip_files:
            print(f"Downloading {os.path.basename(zip_file)}...")
            files.download(zip_file)
        print("All downloads complete!")
    except ImportError:
        print("\nNot in Colab. Zip files saved to:", output_directory)

Reading labels from Excel files...
  letter: Loaded 1232 rows
    Columns: ['Image Path', 'Label']
  word: Loaded 220 rows
    Columns: ['Image Path', 'Label']
  paragraph: Loaded 22 rows
    Columns: ['Image Path', 'Label']

Found root folders: ['Letters', 'Paragraphs', 'Words']
  Mapping: 'letter' -> 'Letters'
  Mapping: 'paragraph' -> 'Paragraphs'
  Mapping: 'word' -> 'Words'

Total participants (P_001-P_022): ['P_001', 'P_002', 'P_003', 'P_004', 'P_005', 'P_006', 'P_007', 'P_008', 'P_009', 'P_010', 'P_011', 'P_012', 'P_013', 'P_014', 'P_015', 'P_016', 'P_017', 'P_018', 'P_019', 'P_020', 'P_021', 'P_022']
Split distribution: Train=17, Val=2, Test=3

Randomly selected participants:
Train (17): ['P_002', 'P_003', 'P_005', 'P_006', 'P_007', 'P_010', 'P_011', 'P_012', 'P_013', 'P_014', 'P_015', 'P_016', 'P_017', 'P_018', 'P_019', 'P_020', 'P_022']
Val   (2): ['P_008', 'P_009']
Test  (3): ['P_001', 'P_004', 'P_021']

Processing TRAIN set...
  Copied: Letters/P_022_task1 (ID: P_022)
  Cop

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All downloads complete!
